# CLF5 Evaluation — standalone

Dubbing is already done. This notebook only reads `eval_records/` and the
dubbed mp4s, so it loads **no** pipeline models — no NLLB, no CLF5, no Demucs,
no pyannote. Only Whisper, and only for Cell 3.

| Cell | Needs GPU | Roughly |
|---|---|---|
| 1 setup | no | 2 min |
| 2 transcripts | no | seconds |
| 3 re-transcribe dubs | **yes** | 15 min |
| 4 cleanup | no | seconds |
| 5 SpeakerSim | no | 5 min |
| 6 metrics | no | seconds |

Cells 2–6 run fine on a **CPU runtime**. Only Cell 3 needs the T4, so if you're
short on GPU credits, run 3 alone on GPU and switch back.


In [ ]:
# ─────────────────────────────────────────
# CELL 1 — setup (no pipeline models)
# ─────────────────────────────────────────
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

!pip install -q jiwer sacrebleu resemblyzer pandas soundfile

OUTPUT_DIR = "/content/drive/MyDrive/dub_pipeline_assets/outputs"

# ── WHICH ARM ARE YOU EVALUATING? ────────────────────────────────
#   ""            -> the original cloned run   (eval_records/)
#   "_nocloning"  -> the CHANGE 3 baseline     (eval_records_nocloning/)
# Everything below is derived from this one line. Run the whole notebook
# once per arm; the two arms write separate CSVs and never collide.
RUN_SUFFIX = ""

EVAL_DIR = f"{OUTPUT_DIR}/eval_records{RUN_SUFFIX}"
# Baseline dubs live in their own folder; the cloned run's sit in OUTPUT_DIR root.
DUB_DIR  = f"{OUTPUT_DIR}/dubbed{RUN_SUFFIX}" if RUN_SUFFIX else OUTPUT_DIR

import glob
recs = glob.glob(f"{EVAL_DIR}/*_eval.json")
dubs = glob.glob(f"{DUB_DIR}/*_dubbed{RUN_SUFFIX}.mp4")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"RUN_SUFFIX  : {RUN_SUFFIX!r}  ({'no-cloning baseline' if RUN_SUFFIX else 'cloned run'})")
print(f"EVAL_DIR    : {EVAL_DIR}")
print(f"DUB_DIR     : {DUB_DIR}")
print(f"exists      : {os.path.exists(OUTPUT_DIR)}")
print(f"eval records: {len(recs)}")
print(f"dubbed mp4s : {len(dubs)}")
print(f"transcripts : {len(glob.glob(f'{OUTPUT_DIR}/transcripts/*'))}")
if not recs:
    print("\n!! No eval records — check EVAL_DIR points where the pipeline wrote.")


In [ ]:
# ─────────────────────────────────────────
# CELL 2 —: load human transcripts into the eval records
# ─────────────────────────────────────────
# Fills source_ref (and english_ref if the file contains an English section).
# Handles .txt / .srt / .vtt / .json / .csv, strips timecodes and speaker tags,
# and splits by script: Devanagari/Telugu/Tamil -> source_ref, Latin -> english_ref.
import glob, json, os, re, unicodedata

TRANS_SRC = f"{OUTPUT_DIR}/transcripts"

SCRIPT_RANGES = {
    "deva": (0x0900, 0x097F),
    "telu": (0x0C00, 0x0C7F),
    "taml": (0x0B80, 0x0BFF),
}

def _script_of(line):
    counts = {k: 0 for k in SCRIPT_RANGES}
    latin = 0
    for ch in line:
        o = ord(ch)
        for k, (a, b) in SCRIPT_RANGES.items():
            if a <= o <= b:
                counts[k] += 1
        if 'a' <= ch.lower() <= 'z':
            latin += 1
    indic = sum(counts.values())
    if indic == 0 and latin == 0:
        return None
    return "indic" if indic >= latin else "latin"

_TS = re.compile(r"^\s*(\d+\s*$|\d{1,2}:\d{2}(:\d{2})?([.,]\d+)?\s*(-->|-)?)")
_SPK = re.compile(r"^\s*(speaker\s*\d+|spk\s*\d+|[A-Z][a-z]+)\s*:\s*", re.I)

def _clean_lines(raw):
    out = []
    for ln in raw.splitlines():
        ln = ln.strip()
        if not ln or ln.upper() == "WEBVTT":
            continue
        if _TS.match(ln) or "-->" in ln:
            continue
        ln = _SPK.sub("", ln)
        if ln:
            out.append(ln)
    return out

def parse_transcript(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".json":
        d = json.load(open(path, encoding="utf-8"))
        if isinstance(d, dict):
            for k in ("text", "transcript", "transcription"):
                if isinstance(d.get(k), str):
                    return _clean_lines(d[k])
            for k in ("segments", "results", "words"):
                if isinstance(d.get(k), list):
                    return _clean_lines(" ".join(
                        str(s.get("text", "")) for s in d[k] if isinstance(s, dict)))
        elif isinstance(d, list):
            return _clean_lines(" ".join(
                str(s.get("text", s)) for s in d))
        return []
    if ext in (".csv", ".tsv"):
        import csv
        sep = "\t" if ext == ".tsv" else ","
        rows = list(csv.reader(open(path, encoding="utf-8", errors="replace"),
                              delimiter=sep))
        if not rows: return []
        hdr = [h.strip().lower() for h in rows[0]]
        ti = next((i for i, h in enumerate(hdr) if "text" in h or "transcript" in h), None)
        body = rows[1:] if ti is not None else rows
        ti = ti if ti is not None else (len(hdr) - 1)
        return _clean_lines(" ".join(r[ti] for r in body if len(r) > ti))
    raw = open(path, encoding="utf-8", errors="replace").read()
    return _clean_lines(raw)

def split_by_script(lines):
    src, eng = [], []
    for ln in lines:
        s = _script_of(ln)
        (eng if s == "latin" else src).append(ln)
    return " ".join(src).strip(), " ".join(eng).strip()

files = sorted(glob.glob(f"{TRANS_SRC}/*"))
print(f"{len(files)} transcript file(s) in {TRANS_SRC}\n")

updated, no_record, empty = [], [], []
for p in files:
    cid = os.path.splitext(os.path.basename(p))[0]
    rec = f"{EVAL_DIR}/{cid}_eval.json"
    if not os.path.exists(rec):
        no_record.append(cid); continue

    lines = parse_transcript(p)
    src_txt, eng_txt = split_by_script(lines)
    if not src_txt and not eng_txt:
        empty.append(cid); continue

    r = json.load(open(rec, encoding="utf-8"))
    if src_txt:
        r["source_ref"] = src_txt
        r["hindi_ref"]  = src_txt          # legacy field name in the schema
    if eng_txt:
        r["english_ref"] = eng_txt
    r["transcript_file"] = os.path.basename(p)
    json.dump(r, open(rec, "w", encoding="utf-8"), indent=2, ensure_ascii=False)

    updated.append(cid)
    print(f"[ok] {cid:8s} source_ref {len(src_txt):6d} chars | "
          f"english_ref {len(eng_txt):6d} chars")

print(f"\nupdated {len(updated)}")
if no_record: print(f"no eval record for: {no_record}")
if empty:     print(f"parsed empty: {empty}")

n_eng = sum(1 for c in updated
            if json.load(open(f"{EVAL_DIR}/{c}_eval.json",
                              encoding="utf-8")).get("english_ref", "").strip())
print(f"\nclips with english_ref: {n_eng}/{len(updated)}")
if n_eng == 0:
    print("  -> BLEU/chrF cannot be computed. They need a HUMAN English")
    print("     translation. Source-language transcripts alone enable")
    print("     WER_asr/CER_asr but not BLEU/chrF.")


In [ ]:
# ─────────────────────────────────────────
# CELL 3 — re-transcribe the dubs  (ONLY cell needing GPU)
# ─────────────────────────────────────────
# Loads Whisper alone — not the dubbing pipeline. Resumable: each record is
# written as it finishes, so a disconnect costs you at most one clip.
!pip install -q faster-whisper

import glob, json, os, subprocess, tempfile
from faster_whisper import WhisperModel
import torch

dev = "cuda" if torch.cuda.is_available() else "cpu"
ct  = "float16" if dev == "cuda" else "int8"
print(f"device: {dev}")
model = WhisperModel("large-v3", device=dev, compute_type=ct)

recs = sorted(glob.glob(f"{EVAL_DIR}/*_eval.json"))
print(f"{len(recs)} records\n")

for p in recs:
    r = json.load(open(p, encoding="utf-8"))
    cid = r["clip_id"]
    if r.get("dub_hyp"):
        print(f"[skip] {cid}"); continue

    dub = r.get("final_dubbed_audio_path") or f"{DUB_DIR}/{cid}_dubbed{RUN_SUFFIX}.mp4"
    if not os.path.exists(dub):
        print(f"[miss] {cid}"); continue

    with tempfile.TemporaryDirectory() as td:
        w = os.path.join(td, "d.wav")
        subprocess.run(["ffmpeg","-y","-loglevel","error","-i",dub,
                        "-vn","-ac","1","-ar","16000",w], check=True)
        segs, info = model.transcribe(w, language="en", beam_size=5,
                                      condition_on_previous_text=False)
        segs = list(segs)

    r["dub_hyp"] = " ".join(s.text.strip() for s in segs)
    r["dub_hyp_segments"] = [{"start":round(s.start,3),"end":round(s.end,3),
                              "text":s.text.strip()} for s in segs]
    json.dump(r, open(p,"w",encoding="utf-8"), indent=2, ensure_ascii=False)
    print(f"[ok]   {cid}: {len(segs)} segments")

print("\ndone")


In [ ]:
# ─────────────────────────────────────────
# CELL 4 — archive stale records  (RUN BEFORE METRICS)
# ─────────────────────────────────────────
# eval_records/ has accumulated clips from older runs (tel_04 vs tel_4,
# vid_*_input, etc). Averaging over those would corrupt every per-language
# mean in the results table. This inspects first and only moves files.
import glob, os, shutil, json

KEEP = ({f"hin_{i}" for i in [1,2,3,4,18,19,20]} |
        {f"tel_{i}" for i in range(1,8)} |
        {f"tam_{i}" for i in range(1,7)})

recs = sorted(glob.glob(f"{EVAL_DIR}/*_eval.json"))
keep, extra = [], []
for p in recs:
    cid = os.path.basename(p).replace("_eval.json", "")
    (keep if cid in KEEP else extra).append(cid)

print(f"{len(recs)} records\n")
print(f"KEEP  ({len(keep)}): {sorted(keep)}")
print(f"\nEXTRA ({len(extra)}): {sorted(extra)}")
print(f"\nmissing from KEEP: {sorted(KEEP - set(keep))}")

# ---- review the lists above, then set to True and re-run ----
ARCHIVE = False

if ARCHIVE and extra:
    dst = f"{OUTPUT_DIR}/eval_records{RUN_SUFFIX}_archive"
    os.makedirs(dst, exist_ok=True)
    for cid in extra:
        shutil.move(f"{EVAL_DIR}/{cid}_eval.json",
                    f"{dst}/{cid}_eval.json")
    print(f"\narchived {len(extra)} -> {dst}")
elif extra:
    print("\n(set ARCHIVE = True to move these out)")


In [ ]:
# ─────────────────────────────────────────
# CELL 5 —: SpeakerSim  (metric #4)
# ─────────────────────────────────────────
# Cosine similarity between the Resemblyzer embedding of the source speaker's
# persisted reference clip and that same speaker's segments in the dub.
# Requires the CELL 8 patch that copies reference wavs out of the temp dir,
# so it only works for clips dubbed with this notebook.
import glob, json, os, subprocess, tempfile, warnings
warnings.filterwarnings("ignore")
import numpy as np, soundfile as sf
from resemblyzer import VoiceEncoder, preprocess_wav

_enc = VoiceEncoder(verbose=False)

def _embed_file(p):
    try:
        return _enc.embed_utterance(preprocess_wav(p))
    except Exception:
        return None

def _embed_array(y, sr=16000):
    try:
        if np.sqrt(np.mean(y ** 2)) < 1e-4:
            return None
        return _enc.embed_utterance(preprocess_wav(y, source_sr=sr))
    except Exception:
        return None

def _cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

for p in sorted(glob.glob(f"{EVAL_DIR}/*_eval.json")):
    r   = json.load(open(p, encoding="utf-8"))
    cid = r["clip_id"]
    if r.get("speaker_sim") is not None:
        print(f"[skip] {cid}"); continue

    refs = r.get("speaker_refs", {})
    dub  = r.get("final_dubbed_audio_path") or f"{DUB_DIR}/{cid}_dubbed{RUN_SUFFIX}.mp4"
    if not refs or not os.path.exists(dub):
        print(f"[miss] {cid}"); continue

    with tempfile.TemporaryDirectory() as td:
        w = os.path.join(td, "d.wav")
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", dub,
                        "-vn", "-ac", "1", "-ar", "16000", w], check=True)
        y, sr = sf.read(w)
        if y.ndim > 1: y = y.mean(1)
        y = y.astype(np.float32)

        per_spk = {}
        for spk, info in refs.items():
            rp = info.get("ref_path")
            if not rp or not os.path.exists(rp):
                continue
            e_ref = _embed_file(rp)
            if e_ref is None:
                continue
            # concatenate this speaker's slices of the dub
            spans = [s for s in r.get("segments", []) if s.get("speaker") == spk]
            if spans:
                chunks = [y[int(s["start"] * sr):int(s["end"] * sr)] for s in spans]
                chunks = [c for c in chunks if len(c) > sr // 2]
                seg_audio = np.concatenate(chunks) if chunks else y
            else:
                seg_audio = y
            e_dub = _embed_array(seg_audio, sr)
            if e_dub is None:
                continue
            per_spk[spk] = round(_cos(e_ref, e_dub), 4)

    if per_spk:
        # duration-weighted where possible, else plain mean
        wts = {}
        for spk in per_spk:
            wts[spk] = sum(s["slot_sec"] for s in r.get("segments", [])
                           if s.get("speaker") == spk) or 1.0
        tot = sum(wts.values())
        r["speaker_sim"] = round(sum(per_spk[s] * wts[s] for s in per_spk) / tot, 4)
    else:
        r["speaker_sim"] = None
    r["speaker_sim_per_speaker"] = per_spk

    json.dump(r, open(p, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    print(f"[ok]   {cid}: SpeakerSim {r['speaker_sim']}  {per_spk}")


In [ ]:
# ─────────────────────────────────────────
# CELL 6 —: compute all nine metrics
# ─────────────────────────────────────────
#  1. WER_asr / CER_asr   pipeline_src  vs source_ref     (ASR accuracy)
#  2. BLEU / chrF         pipeline_eng  vs english_ref    (translation quality)
#  3. WER_tts / CER_tts   dub_hyp       vs pipeline_eng   (TTS intelligibility)
#  4. SpeakerSim          from CELL 14
#  5. Coverage            from the pipeline
#  6. RTF                 from the pipeline
import glob, json, re
import pandas as pd, jiwer, sacrebleu

def _n(t):
    t = str(t or "").lower()
    t = re.sub(r"[^\w\s]", " ", t)
    return " ".join(t.split())

def _wer(ref, hyp):
    ref, hyp = _n(ref), _n(hyp)
    if not ref or not hyp: return None
    return round(jiwer.wer(ref, hyp), 4)

def _cer(ref, hyp):
    ref, hyp = _n(ref), _n(hyp)
    if not ref or not hyp: return None
    return round(jiwer.cer(ref, hyp), 4)

LANGNAME = {"hin": "Hindi", "tel": "Telugu", "tam": "Tamil"}

rows = []
for p in sorted(glob.glob(f"{EVAL_DIR}/*_eval.json")):
    r   = json.load(open(p, encoding="utf-8"))
    cid = r["clip_id"]

    src_ref = r.get("source_ref") or r.get("hindi_ref") or ""
    eng_ref = r.get("english_ref") or ""
    pipe_src = r.get("source_asr_text") or ""
    pipe_eng = r.get("english_mt_text") or ""
    dub_hyp  = r.get("dub_hyp") or ""

    bleu = chrf = None
    if pipe_eng.strip() and eng_ref.strip():
        bleu = round(sacrebleu.sentence_bleu(pipe_eng, [eng_ref]).score, 2)
        chrf = round(sacrebleu.sentence_chrf(pipe_eng, [eng_ref]).score, 2)

    rows.append({
        "clip":      cid,
        "lang":      LANGNAME.get(cid.split("_")[0], r.get("src_language", "?")),
        "WER_asr":   _wer(src_ref, pipe_src),
        "CER_asr":   _cer(src_ref, pipe_src),
        "BLEU":      bleu,
        "chrF":      chrf,
        "WER_tts":   _wer(pipe_eng, dub_hyp),
        "CER_tts":   _cer(pipe_eng, dub_hyp),
        "SpeakerSim": r.get("speaker_sim"),
        "Coverage_%": r.get("coverage_pct"),
        "RTF":       round(r.get("rtf", 0), 2) if r.get("rtf") else None,
    })

df = pd.DataFrame(rows).sort_values(["lang", "clip"]).reset_index(drop=True)
METRICS = ["WER_asr","CER_asr","BLEU","chrF","WER_tts","CER_tts",
           "SpeakerSim","Coverage_%","RTF"]

pd.set_option("display.width", 220)
print("PER-CLIP")
print("=" * 118)
print(df.to_string(index=False))

# ── Aggregation: Mean (all) then per language, per the metrics spec ──
agg = [df[METRICS].mean().round(3).to_dict() | {"clip": "Mean (all)", "lang": ""}]
for lg in ["Hindi", "Telugu", "Tamil"]:
    sub = df[df["lang"] == lg]
    if len(sub):
        agg.append(sub[METRICS].mean().round(3).to_dict()
                   | {"clip": f"Mean ({lg})", "lang": ""})

agg_df = pd.DataFrame(agg)[["clip", "lang"] + METRICS]
print("\nAGGREGATE")
print("=" * 118)
print(agg_df.to_string(index=False))

final = pd.concat([df, agg_df], ignore_index=True)
final.to_csv(f"{OUTPUT_DIR}/eval_metrics_CLF5{RUN_SUFFIX}.csv", index=False)
print(f"\nSaved: {OUTPUT_DIR}/eval_metrics_CLF5{RUN_SUFFIX}.csv")

# corpus BLEU — cite this, not the mean of per-clip BLEU
hyps, refs = [], []
for p in sorted(glob.glob(f"{EVAL_DIR}/*_eval.json")):
    r = json.load(open(p, encoding="utf-8"))
    if (r.get("english_mt_text") or "").strip() and (r.get("english_ref") or "").strip():
        hyps.append(r["english_mt_text"]); refs.append(r["english_ref"])
if hyps:
    print(f"\nCORPUS BLEU {sacrebleu.corpus_bleu(hyps,[refs]).score:.2f}"
          f"  chrF {sacrebleu.corpus_chrf(hyps,[refs]).score:.2f}  (n={len(hyps)})")
else:
    print("\nCORPUS BLEU: not computable — no clip has a human english_ref.")

missing = {m: int(df[m].isna().sum()) for m in METRICS if df[m].isna().any()}
if missing:
    print(f"\nMissing values: {missing}")


In [ ]:
# ─────────────────────────────────────────
# CELL 7 —: export English references to fill in (for BLEU/chrF)
# ─────────────────────────────────────────
# Only needed if your transcripts were source-language only. BLEU and chrF
# cannot be computed without a HUMAN English translation — an MT output or a
# re-ASR of the dub will not do, since the same error then appears on both
# sides of the comparison and the score cannot detect it.
import glob, json
import pandas as pd

rows = []
for p in sorted(glob.glob(f"{EVAL_DIR}/*_eval.json")):
    r = json.load(open(p, encoding="utf-8"))
    rows.append({
        "clip_id":     r["clip_id"],
        "lang":        r.get("src_language", "?"),
        "source_ref":  (r.get("source_ref") or r.get("hindi_ref") or "")[:32000],
        "english_ref": r.get("english_ref", ""),      # <-- FILL THIS
        "notes":       "",
    })
out = f"{OUTPUT_DIR}/english_refs_to_fill.csv"
pd.DataFrame(rows).to_csv(out, index=False)
print(f"{len(rows)} rows -> {out}")
print("Translate source_ref -> english_ref by hand, save as")
print(f"{OUTPUT_DIR}/english_refs_filled.csv, then run CELL 17.")


In [ ]:
# ─────────────────────────────────────────
# CELL 8 — load filled English references back in
# ─────────────────────────────────────────
import pandas as pd, json, os

FILLED = f"{OUTPUT_DIR}/english_refs_filled.csv"
if not os.path.exists(FILLED):
    print(f"Not found: {FILLED}")
else:
    df = pd.read_csv(FILLED).fillna("")
    n = 0
    for _, row in df.iterrows():
        ref = str(row.get("english_ref", "")).strip()
        if not ref: continue
        p = f"{EVAL_DIR}/{row['clip_id']}_eval.json"
        if not os.path.exists(p):
            print(f"  no record: {row['clip_id']}"); continue
        r = json.load(open(p, encoding="utf-8"))
        r["english_ref"] = ref
        json.dump(r, open(p, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
        n += 1
    print(f"Updated {n} record(s). Re-run CELL 15 to get BLEU/chrF.")


---

## CHANGE 3 — cloned vs no-cloning comparison

Run the notebook twice first: once with `RUN_SUFFIX = ""`, once with
`RUN_SUFFIX = "_nocloning"`. Both arms need cells 2–6 completed so that
`dub_hyp`, `speaker_sim` and the metrics exist in each record.

The cell below reads both record directories, pairs them by clip, and runs
the paired tests. It reports only clips present in **both** arms — an
unpaired mean difference is not the comparison you want to publish.


In [ ]:
# ─────────────────────────────────────────
# CELL 9 — CLONED vs NO-CLONING  (CHANGE 3 deliverable)
# ─────────────────────────────────────────
# Paired comparison. Reads eval_records/ and eval_records_nocloning/, keeps
# only clips present in both, and reports per-clip deltas plus paired tests.
#
# Two tests per metric, deliberately:
#   paired t-test  — parametric, what most reviewers expect
#   Wilcoxon       — signed-rank, no normality assumption, honest at n=18
# Report both. If they disagree, trust Wilcoxon and say so.
#
# Cohen's dz is the paired effect size. Report it next to p — a p-value at
# n=18 tells a reviewer almost nothing on its own.
import glob, json, os
import numpy as np, pandas as pd
from scipy import stats

CLONED_DIR   = f"{OUTPUT_DIR}/eval_records"
BASELINE_DIR = f"{OUTPUT_DIR}/eval_records_nocloning"

def _load(d, arm):
    out = {}
    for p in sorted(glob.glob(f"{d}/*_eval.json")):
        r = json.load(open(p, encoding="utf-8"))
        out[r["clip_id"]] = r
    print(f"{arm:9s}: {len(out)} records  ({d})")
    return out

A = _load(CLONED_DIR,   "cloned")
B = _load(BASELINE_DIR, "baseline")

paired = sorted(set(A) & set(B))
print(f"\npaired clips: {len(paired)}")
if set(A) - set(B):
    print(f"  cloned only  : {sorted(set(A) - set(B))}")
if set(B) - set(A):
    print(f"  baseline only: {sorted(set(B) - set(A))}")
assert paired, "No clips in both arms — has the baseline run finished?"

# ── integrity check: did the freeze actually hold? ──
# If english_mt_text differs between arms, WER_tts is being computed against
# two different references and the intelligibility comparison is invalid.
def _mt(r): return (r.get("english_mt_text") or "").strip()
drift = [c for c in paired if _mt(A[c]) != _mt(B[c])]
seg_drift = [c for c in paired
             if len(A[c].get("segments", [])) != len(B[c].get("segments", []))]
print("\nFREEZE INTEGRITY")
print(f"  MT text differs      : {len(drift)}/{len(paired)} "
      f"{drift if drift else '✅'}")
print(f"  segment count differs: {len(seg_drift)}/{len(paired)} "
      f"{seg_drift if seg_drift else '✅'}")
if drift or seg_drift:
    print("  ⚠ These clips are NOT a clean paired comparison. Either rebuild "
          "the freeze cache (CELL 8b) and re-run the baseline for them, or "
          "exclude them and say so in the paper.")

LANGNAME = {"hin": "Hindi", "tel": "Telugu", "tam": "Tamil"}
COMPARE  = ["WER_tts", "CER_tts", "SpeakerSim", "Coverage_%", "RTF"]

def _metrics(r):
    import re, jiwer
    def _n(t):
        t = str(t or "").lower()
        return " ".join(re.sub(r"[^\w\s]", " ", t).split())
    ref, hyp = _n(r.get("english_mt_text")), _n(r.get("dub_hyp"))
    return {
        "WER_tts":    round(jiwer.wer(ref, hyp), 4) if ref and hyp else None,
        "CER_tts":    round(jiwer.cer(ref, hyp), 4) if ref and hyp else None,
        "SpeakerSim": r.get("speaker_sim"),
        "Coverage_%": r.get("coverage_pct"),
        "RTF":        round(r.get("rtf"), 3) if r.get("rtf") else None,
    }

rows = []
for c in paired:
    ma, mb = _metrics(A[c]), _metrics(B[c])
    row = {"clip": c, "lang": LANGNAME.get(c.split("_")[0], "?")}
    for m in COMPARE:
        row[f"{m}_cloned"] = ma[m]
        row[f"{m}_base"]   = mb[m]
        row[f"{m}_delta"]  = (round(ma[m] - mb[m], 4)
                              if ma[m] is not None and mb[m] is not None else None)
    rows.append(row)

df = pd.DataFrame(rows).sort_values(["lang", "clip"]).reset_index(drop=True)
pd.set_option("display.width", 250)

print("\nPER-CLIP  (delta = cloned - baseline)")
print("=" * 130)
for m in COMPARE:
    cols = ["clip", "lang", f"{m}_cloned", f"{m}_base", f"{m}_delta"]
    print(f"\n{m}")
    print(df[cols].to_string(index=False))

# ── paired statistics ──
print("\n\nPAIRED TESTS  (n = clips with both values)")
print("=" * 130)
stat_rows = []
for m in COMPARE:
    a = df[f"{m}_cloned"].to_numpy(dtype=float)
    b = df[f"{m}_base"].to_numpy(dtype=float)
    ok = ~(np.isnan(a) | np.isnan(b))
    a, b = a[ok], b[ok]
    if len(a) < 3:
        stat_rows.append({"metric": m, "n": len(a)}); continue
    d  = a - b
    sd = d.std(ddof=1)
    t_stat, t_p = stats.ttest_rel(a, b)
    try:
        w_stat, w_p = stats.wilcoxon(a, b)
    except ValueError:                       # all differences zero
        w_stat, w_p = float("nan"), 1.0
    # 95% CI on the mean paired difference
    se = sd / np.sqrt(len(d)) if len(d) else float("nan")
    tc = stats.t.ppf(0.975, len(d) - 1)
    stat_rows.append({
        "metric":     m,
        "n":          len(a),
        "cloned":     round(float(a.mean()), 4),
        "baseline":   round(float(b.mean()), 4),
        "delta":      round(float(d.mean()), 4),
        "CI95_lo":    round(float(d.mean() - tc * se), 4),
        "CI95_hi":    round(float(d.mean() + tc * se), 4),
        "t":          round(float(t_stat), 3),
        "p_ttest":    float(f"{t_p:.2e}"),
        "p_wilcoxon": float(f"{w_p:.2e}"),
        "cohen_dz":   round(float(d.mean() / sd), 3) if sd > 0 else None,
    })

stats_df = pd.DataFrame(stat_rows)
print(stats_df.to_string(index=False))

out_pairs = f"{OUTPUT_DIR}/change3_cloned_vs_nocloning_perclip.csv"
out_stats = f"{OUTPUT_DIR}/change3_cloned_vs_nocloning_stats.csv"
df.to_csv(out_pairs, index=False)
stats_df.to_csv(out_stats, index=False)
print(f"\nSaved: {out_pairs}")
print(f"Saved: {out_stats}")

# ── plain-language read-out ──
print("\n\nREAD-OUT")
print("=" * 130)
_g = {r["metric"]: r for r in stat_rows if "delta" in r}
if "SpeakerSim" in _g:
    r = _g["SpeakerSim"]
    print(f"SpeakerSim : cloned {r['cloned']:.3f} vs generic {r['baseline']:.3f} "
          f"(delta {r['delta']:+.3f}, p={r['p_wilcoxon']:.1e}, dz={r['cohen_dz']})")
    print(f"             -> the generic floor is {r['baseline']:.3f}. This is what "
          f"makes {r['cloned']:.3f} interpretable.")
if "WER_tts" in _g:
    r = _g["WER_tts"]
    direction = ("cloning is WORSE" if r["delta"] > 0 else
                 "cloning is BETTER" if r["delta"] < 0 else "identical")
    print(f"WER_tts    : cloned {r['cloned']:.3f} vs generic {r['baseline']:.3f} "
          f"(delta {r['delta']:+.3f}, p={r['p_wilcoxon']:.1e}) — {direction}")
    print(f"             CI95 [{r['CI95_lo']:+.3f}, {r['CI95_hi']:+.3f}]. If this "
          f"interval sits inside your TOST margin (change 2), you can state "
          f"equivalence rather than 'no significant difference'.")
print("\nThe claim the paper wants is: SpeakerSim rises sharply while WER_tts "
      "does not move. Check both lines above before writing that sentence.")
